# Formula SAE optimal lap-time simulation

Run a minimum-lap-time simulation for the FRUCD electric FSAE car on the Vendrell kart circuit. The workflow mirrors the F1 example while using an autocross-scale track and the fitted Hoosier tire model.

## 1. Locate the runtime and input data

In [1]:
import os
import sys
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root(start=Path.cwd()):
    for directory in (start, *start.parents):
        if (directory / 'database' / 'vehicles' / 'fsae').is_dir():
            return directory
    raise FileNotFoundError('Start Jupyter inside the fastest-lap repository.')


REPO_ROOT = find_repository_root()
default_python_dir = (
    REPO_ROOT / 'build' / 'artifacts' /
    'fastestlap-python-windows-mingw64-x86_64-fsae-electric' / 'python'
)
python_dir = Path(os.environ.get('FASTESTLAP_PYTHON_DIR', default_python_dir))
if python_dir.is_dir():
    sys.path.insert(0, str(python_dir))

import fastest_lap

required_api = ('create_vehicle_from_xml', 'create_track_from_xml', 'optimal_laptime')
missing_api = [name for name in required_api if not hasattr(fastest_lap, name)]
if missing_api:
    raise RuntimeError(f'Missing fastest_lap API: {missing_api}; check FASTESTLAP_PYTHON_DIR.')
print(f'fastest_lap: {Path(fastest_lap.__file__).resolve()}')
print(f'repository:  {REPO_ROOT}')

fastest_lap: D:\projects\fastest-lap\build\artifacts\fastestlap-python-windows-mingw64-x86_64-fsae-electric\python\fastest_lap.py
repository:  D:\projects\fastest-lap


## 2. Load the FSAE vehicle

The repository XML contains a relocatable tire filename. A temporary copy receives the absolute MAT-file path so the source XML remains unchanged.

In [2]:
fsae_database = REPO_ROOT / 'database' / 'vehicles' / 'fsae'
source_vehicle_xml = fsae_database / 'fsae-2026-pacejka-simple.xml'
tire_filename = 'Hoosier_R20_16(18)x75(60)-10x8(7).mat'
tire_file = Path(os.environ.get(
    'FASTESTLAP_FSAE_TIRE_FILE', fsae_database / tire_filename
)).expanduser().resolve()
if not tire_file.is_file():
    raise FileNotFoundError(
        f'Tire data not found at {tire_file}; set FASTESTLAP_FSAE_TIRE_FILE.'
    )

temporary_directory = tempfile.TemporaryDirectory(prefix='fastestlap-fsae-lts-')
runnable_vehicle_xml = Path(temporary_directory.name) / source_vehicle_xml.name
tree = ET.parse(source_vehicle_xml)
mat_nodes = tree.getroot().findall('.//mat-file')
if not mat_nodes:
    raise ValueError(f'No mat-file nodes found in {source_vehicle_xml}')
for node in mat_nodes:
    node.text = str(tire_file)
tree.write(runnable_vehicle_xml, encoding='utf-8', xml_declaration=True)

vehicle = 'fsae-electric-lts-example'
fastest_lap.create_vehicle_from_xml(vehicle, str(runnable_vehicle_xml))
print(f'vehicle type: {fastest_lap.variable_type(vehicle)}')
print(f'tire data:   {tire_file}')

ValueError: No mat-file nodes found in D:\projects\fastest-lap\database\vehicles\fsae\fsae-2026-pacejka-simple.xml

## 3. Load the track and choose the optimization mesh

In [3]:
track = 'vendrell-fsae-lts-example'
track_xml = REPO_ROOT / 'database' / 'tracks' / 'vendrell' / 'vendrell.xml'
fastest_lap.create_track_from_xml(track, str(track_xml))
s_track = np.asarray(fastest_lap.track_download_data(track, 'arclength'))

# Use 250 nodes for a practical example; use 1 for the full 500-node mesh.
mesh_stride = 2
s_mesh = s_track[::mesh_stride]
print(f'track length: {s_track[-1]:.1f} m; optimization nodes: {len(s_mesh)}')

track length: 1250.7 m; optimization nodes: 250


## 4. Solve the minimum-lap-time problem

Positive `chassis.throttle` means drive torque and negative values mean braking. Solver progress is enabled because this is the longest cell.

In [4]:
options = '''
<options>
  <output_variables>
    <prefix>fsae_lts_run/</prefix>
  </output_variables>
  <print_level>5</print_level>
</options>
'''

result_names = fastest_lap.optimal_laptime(vehicle, track, s_mesh, options)
run = fastest_lap.download_variables(*result_names)

s = np.asarray(run['road.arclength'])
x = np.asarray(run['chassis.position.x'])
y = np.asarray(run['chassis.position.y'])
speed_kmh = 3.6 * np.asarray(run['chassis.velocity.x'])
steering_deg = np.rad2deg(np.asarray(run['front-axle.steering-angle']))
throttle = np.asarray(run['chassis.throttle'])
time = np.asarray(run['time'])

lap_time = float(time[-1])
print(f'Optimal lap time: {lap_time:.3f} s')
print(f'Speed range: {speed_kmh.min():.1f} - {speed_kmh.max():.1f} km/h')

OSError: [WinError -1073741569] Windows Error 0xc00000ff

## 5. Inspect the racing line and driver controls

In [ ]:
fig, axis = plt.subplots(figsize=(8, 7), constrained_layout=True)
points = axis.scatter(x, y, c=speed_kmh, s=16, cmap='viridis')
axis.plot(x, y, color='black', linewidth=0.5, alpha=0.45)
axis.set(xlabel='x [m]', ylabel='y [m]', title=f'FSAE optimal line — {lap_time:.3f} s')
axis.set_aspect('equal', adjustable='box')
axis.grid(True, alpha=0.2)
fig.colorbar(points, ax=axis, label='Speed [km/h]')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
axes[0].plot(s, speed_kmh, color='#2667ff')
axes[0].set_ylabel('Speed [km/h]')
axes[1].plot(s, steering_deg, color='#f28e2b')
axes[1].axhline(0.0, color='black', linewidth=0.6)
axes[1].set_ylabel('Steering [deg]')
axes[2].fill_between(s, 0.0, np.maximum(throttle, 0.0), label='Drive', color='#2ca02c', alpha=0.75)
axes[2].fill_between(s, 0.0, np.minimum(throttle, 0.0), label='Brake', color='#d62728', alpha=0.75)
axes[2].set(xlabel='Arclength [m]', ylabel='Control [-]')
axes[2].legend(loc='upper right')
for axis in axes:
    axis.grid(True, alpha=0.25)
plt.show()

## 6. Clean up

Run this after inspecting the results. It removes example objects from the fastest-lap registry and deletes the temporary XML.

In [ ]:
fastest_lap.delete_variable('fsae_lts_run/*')
fastest_lap.delete_variable(vehicle)
fastest_lap.delete_variable(track)
temporary_directory.cleanup()
print('FSAE LTS example objects removed.')